In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.table(
    "INSURANCE_ML_DB.INSURANCE_SCHEMA.INSURANCE_DATA"
)

print("Total records:", df.count())

df.show(10)

In [ ]:
import pandas as pd

insurance_df = df.to_pandas()

print("Dataset shape:", insurance_df.shape)
insurance_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = insurance_df.drop("CHARGES", axis=1)
y = insurance_df["CHARGES"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

In [ ]:
%pip install xgboost scikit-learn

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

In [ ]:
numeric_features = ["AGE", "BMI", "CHILDREN"]
categorical_features = ["SEX", "SMOKER", "REGION"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
            objective="reg:squarederror"
        ))
    ]
)

print("XGBoost model created successfully!")

In [ ]:
model.fit(X_train, y_train)

print("Model training completed successfully!")

In [ ]:
# Predict insurance charges for test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("MODEL EVALUATION RESULTS")
print("------------------------")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))
print("R2 Accuracy:", round(r2 * 100, 2), "%")

In [ ]:
comparison = pd.DataFrame({
    "Actual_Charges": y_test.values,
    "Predicted_Charges": y_pred
})

comparison["Difference"] = abs(
    comparison["Actual_Charges"] -
    comparison["Predicted_Charges"]
)

comparison.head(10)

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

print("Snowflake Model Registry imported successfully!")

In [ ]:
session.sql(
    "USE DATABASE INSURANCE_ML_DB"
).collect()

session.sql(
    "USE SCHEMA INSURANCE_SCHEMA"
).collect()

session.sql(
    "USE WAREHOUSE COMPUTE_WH"
).collect()

print("Database, schema and warehouse selected successfully!")

In [ ]:
print(session.get_current_database())
print(session.get_current_schema())
print(session.get_current_warehouse())

In [ ]:
session.sql("""
SHOW MODELS IN SCHEMA
INSURANCE_ML_DB.INSURANCE_SCHEMA
""").show()

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

# Create Model Registry
registry = Registry(
    session=session,
    database_name="INSURANCE_ML_DB",
    schema_name="INSURANCE_SCHEMA"
)

# Register trained model
model_version = registry.log_model(
    model=model,
    model_name="INSURANCE_CHARGES_XGB",
    version_name="V1",
    sample_input_data=X_train,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2_SCORE": float(r2)
    },
    comment="XGBoost insurance premium prediction model",
    task=task.Task.TABULAR_REGRESSION
)

print("Model registered successfully!")
print(model_version)

In [ ]:
session.sql("""
SHOW MODELS IN SCHEMA
INSURANCE_ML_DB.INSURANCE_SCHEMA
""").show()

In [ ]:
# Load registered model
loaded_model = registry.get_model(
    "INSURANCE_CHARGES_XGB"
)

# Get version V1
loaded_version = loaded_model.version("V1")

print("Registered model loaded successfully!")
print(loaded_version)

In [ ]:
# Generate predictions using the registered model
predictions = loaded_version.run(
    X_test,
    function_name="predict"
)

print(predictions.head(10))

In [ ]:
# Prepare prediction results
prediction_df = X_test.copy()

prediction_df["ACTUAL_CHARGES"] = y_test.values
prediction_df["PREDICTED_CHARGES"] = predictions["output_feature_0"].values

prediction_df["PREDICTION_DIFFERENCE"] = abs(
    prediction_df["ACTUAL_CHARGES"] -
    prediction_df["PREDICTED_CHARGES"]
)

# Display first 10 rows
print(prediction_df.head(10))

In [ ]:
# Save prediction DataFrame to Snowflake
session.write_pandas(
    prediction_df,
    table_name="INSURANCE_PREDICTIONS",
    database="INSURANCE_ML_DB",
    schema="INSURANCE_SCHEMA",
    auto_create_table=True,
    overwrite=True
)

print("Predictions saved successfully in Snowflake!")

In [ ]:
session.sql("""
SELECT *
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.INSURANCE_PREDICTIONS
LIMIT 10
""").show()

In [ ]:
session.sql("""
SELECT
    COUNT(*) AS TOTAL_PREDICTIONS,
    ROUND(AVG(PREDICTION_DIFFERENCE), 2) AS AVG_PREDICTION_ERROR,
    ROUND(AVG(ACTUAL_CHARGES), 2) AS AVG_ACTUAL_CHARGES,
    ROUND(AVG(PREDICTED_CHARGES), 2) AS AVG_PREDICTED_CHARGES
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.INSURANCE_PREDICTIONS
""").show()

In [ ]:
session.sql("""
CREATE TABLE IF NOT EXISTS
INSURANCE_ML_DB.INSURANCE_SCHEMA.NEW_INSURANCE_CUSTOMERS (
    AGE INTEGER,
    SEX VARCHAR(10),
    BMI FLOAT,
    CHILDREN INTEGER,
    SMOKER VARCHAR(10),
    REGION VARCHAR(30)
)
""").collect()

print("New customer table created successfully!")

In [ ]:
session.sql("""
CREATE STREAM IF NOT EXISTS
INSURANCE_ML_DB.INSURANCE_SCHEMA.INSURANCE_CUSTOMER_STREAM
ON TABLE INSURANCE_ML_DB.INSURANCE_SCHEMA.NEW_INSURANCE_CUSTOMERS
""").collect()

print("Insurance customer stream created successfully!")

In [ ]:
session.sql("""
INSERT INTO INSURANCE_ML_DB.INSURANCE_SCHEMA.NEW_INSURANCE_CUSTOMERS
(AGE, SEX, BMI, CHILDREN, SMOKER, REGION)
VALUES
(25, 'female', 27.5, 0, 'no', 'southwest'),
(40, 'male', 32.1, 2, 'yes', 'southeast'),
(35, 'female', 24.8, 1, 'no', 'northwest')
""").collect()

print("3 new customer records inserted successfully!")

In [ ]:
session.sql("""
SELECT *
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.NEW_INSURANCE_CUSTOMERS
""").show()

In [ ]:
session.sql("""
SELECT *
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.INSURANCE_CUSTOMER_STREAM
""").show()

In [ ]:
# Create a temporary table for the new customer batch
session.sql("""
CREATE OR REPLACE TEMP TABLE INSURANCE_STREAM_BATCH AS
SELECT AGE, SEX, BMI, CHILDREN, SMOKER, REGION
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.NEW_INSURANCE_CUSTOMERS
WHERE 1 = 0
""").collect()

# Capture and consume the new records from the Stream
session.sql("""
INSERT INTO INSURANCE_STREAM_BATCH
SELECT AGE, SEX, BMI, CHILDREN, SMOKER, REGION
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.INSURANCE_CUSTOMER_STREAM
WHERE METADATA$ACTION = 'INSERT'
AND METADATA$ISUPDATE = FALSE
""").collect()

print("New customer batch captured successfully!")

In [ ]:
# Read new customer records
new_customers_df = session.table(
    "INSURANCE_STREAM_BATCH"
).to_pandas()

print("New customer records:")
print(new_customers_df)

print("Total new customers:", len(new_customers_df))

In [ ]:
# Generate predictions using registered model
new_predictions = loaded_version.run(
    new_customers_df,
    function_name="predict"
)

# Display predictions
print("Insurance Premium Predictions:")
print(new_predictions)

In [ ]:
session.sql("""
CREATE TABLE IF NOT EXISTS
INSURANCE_ML_DB.INSURANCE_SCHEMA.AUTOMATED_INSURANCE_PREDICTIONS (
    AGE INTEGER,
    SEX VARCHAR(10),
    BMI FLOAT,
    CHILDREN INTEGER,
    SMOKER VARCHAR(10),
    REGION VARCHAR(30),
    PREDICTED_CHARGES FLOAT,
    PREDICTION_DATE TIMESTAMP_NTZ
)
""").collect()

print("Automated prediction table created successfully!")

In [ ]:
# Combine customer information and predicted charges
automated_results = new_customers_df.copy()

automated_results["PREDICTED_CHARGES"] = (
    new_predictions["output_feature_0"].values
)

# Add prediction timestamp
automated_results["PREDICTION_DATE"] = (
    __import__("pandas").Timestamp.now()
)

print(automated_results)

In [ ]:
session.write_pandas(
    automated_results,
    table_name="AUTOMATED_INSURANCE_PREDICTIONS",
    database="INSURANCE_ML_DB",
    schema="INSURANCE_SCHEMA",
    auto_create_table=False,
    overwrite=False
)

print("Automated predictions saved successfully!")

In [ ]:
session.sql("""
SELECT
    AGE,
    SEX,
    PREDICTED_CHARGES,
    TO_VARCHAR(PREDICTION_DATE) AS PREDICTION_DATE_TEXT
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.AUTOMATED_INSURANCE_PREDICTIONS
""").show()

In [ ]:
session.sql("""
UPDATE INSURANCE_ML_DB.INSURANCE_SCHEMA.AUTOMATED_INSURANCE_PREDICTIONS
SET PREDICTION_DATE = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
""").collect()

print("Prediction timestamps corrected successfully!")

In [ ]:
session.sql("""
SELECT
    AGE,
    SEX,
    PREDICTED_CHARGES,
    TO_VARCHAR(PREDICTION_DATE) AS PREDICTION_DATE
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.AUTOMATED_INSURANCE_PREDICTIONS
""").show()

In [ ]:
import pandas as pd

automated_results = new_customers_df.copy()

automated_results["PREDICTED_CHARGES"] = (
    new_predictions["output_feature_0"].values
)

# Store timestamp as text temporarily
automated_results["PREDICTION_DATE"] = (
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
)

print(automated_results)

In [ ]:
session.sql("""
SELECT
    COUNT(*) AS TOTAL_CUSTOMERS,
    ROUND(AVG(PREDICTED_CHARGES), 2) AS AVG_PREDICTED_PREMIUM,
    MIN(PREDICTION_DATE) AS FIRST_PREDICTION,
    MAX(PREDICTION_DATE) AS LAST_PREDICTION
FROM INSURANCE_ML_DB.INSURANCE_SCHEMA.AUTOMATED_INSURANCE_PREDICTIONS
""").show()

In [ ]:
import snowflake.snowpark
import snowflake.ml

print("Snowpark version:", snowflake.snowpark.__version__)
print("Snowflake ML is available!")